<a href="https://colab.research.google.com/github/Thorfast191/Monocular-Metric-Depth-Estimation/blob/main/MMDE_V1_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
!pip install kaggle pandas tqdm opencv-python

In [18]:
%cd /content/drive/MyDrive/Datasets/NYU Depth V2/nyu_data/data
!ls

/content/drive/MyDrive/Datasets/NYU Depth V2/nyu_data/data
nyu2_test  nyu2_test.csv  nyu2_train  nyu2_train.csv


In [19]:
import os

# ✅ IMPORTANT: root should STOP before "data"
root = "/content/drive/MyDrive/Datasets/NYU Depth V2/nyu_data"

train_csv = os.path.join(root, "data/nyu2_train.csv")
test_csv  = os.path.join(root, "data/nyu2_test.csv")

In [20]:
import pandas as pd
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset

class NYUDataset(Dataset):
    def __init__(self, root_dir, csv_file):
        self.root_dir = root_dir

        # ✅ FIX: CSV has no header
        self.data = pd.read_csv(csv_file, header=None)
        self.data.columns = ["image", "depth"]

        print("Dataset Loaded:", csv_file)
        print(self.data.head())

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # ✅ FIX: correct path joining
        img_path = os.path.join(self.root_dir, row["image"])
        depth_path = os.path.join(self.root_dir, row["depth"])

        # Load image
        img = cv2.imread(img_path)
        if img is None:
            raise ValueError(f"Image not found: {img_path}")

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (320, 240)) / 255.0

        # Load depth
        depth = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED)
        if depth is None:
            raise ValueError(f"Depth not found: {depth_path}")

        depth = cv2.resize(depth, (320, 240)).astype(np.float32)

        # ✅ NYU depth is in mm → convert to meters
        depth = depth / 1000.0

        img = torch.tensor(img).permute(2, 0, 1).float()
        depth = torch.tensor(depth).unsqueeze(0)

        return img, depth

In [21]:
from torch.utils.data import DataLoader

train_dataset = NYUDataset(root, train_csv)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

img, depth = train_dataset[0]

print("Image shape:", img.shape)
print("Depth shape:", depth.shape)
print("Depth min:", depth.min())
print("Depth max:", depth.max())

Dataset Loaded: /content/drive/MyDrive/Datasets/NYU Depth V2/nyu_data/data/nyu2_train.csv
                                          image  \
0   data/nyu2_train/living_room_0038_out/37.jpg   
1  data/nyu2_train/living_room_0038_out/115.jpg   
2    data/nyu2_train/living_room_0038_out/6.jpg   
3   data/nyu2_train/living_room_0038_out/49.jpg   
4  data/nyu2_train/living_room_0038_out/152.jpg   

                                          depth  
0   data/nyu2_train/living_room_0038_out/37.png  
1  data/nyu2_train/living_room_0038_out/115.png  
2    data/nyu2_train/living_room_0038_out/6.png  
3   data/nyu2_train/living_room_0038_out/49.png  
4  data/nyu2_train/living_room_0038_out/152.png  
Image shape: torch.Size([3, 240, 320])
Depth shape: torch.Size([1, 240, 320])
Depth min: tensor(0.0310)
Depth max: tensor(0.2540)


In [22]:
import torch.nn as nn
import torchvision.models as models

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights="IMAGENET1K_V1")

        self.layer0 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.layer1 = nn.Sequential(resnet.maxpool, resnet.layer1)
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

    def forward(self, x):
        x0 = self.layer0(x)
        x1 = self.layer1(x0)
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        return x0, x1, x2, x3, x4

In [23]:
class UpBlock(nn.Module):
    def __init__(self, in_c, skip_c, out_c):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.conv = nn.Sequential(
            nn.Conv2d(in_c + skip_c, out_c, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

In [24]:
class DepthModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()

        self.up4 = UpBlock(2048, 1024, 1024)
        self.up3 = UpBlock(1024, 512, 512)
        self.up2 = UpBlock(512, 256, 256)
        self.up1 = UpBlock(256, 64, 128)

        self.final = nn.Sequential(
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 1, 1)
        )

    def forward(self, x):
        x0, x1, x2, x3, x4 = self.encoder(x)

        d4 = self.up4(x4, x3)
        d3 = self.up3(d4, x2)
        d2 = self.up2(d3, x1)
        d1 = self.up1(d2, x0)

        return self.final(d1)

In [25]:
class SILogLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, pred, target):
        mask = target > 0

        pred = pred[mask]
        target = target[mask]

        pred = torch.clamp(pred, min=1e-3)

        log_diff = torch.log(pred) - torch.log(target + 1e-6)

        return torch.sqrt((log_diff**2).mean() - 0.85 * (log_diff.mean()**2))

In [26]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = DepthModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = SILogLoss()

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 149MB/s]


In [27]:
from tqdm import tqdm
import torch

save_path = "/content/drive/MyDrive/checkpoints"
os.makedirs(save_path, exist_ok=True)

scaler = torch.cuda.amp.GradScaler()

for epoch in range(5):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader)

    for img, depth in loop:
        img, depth = img.to(device), depth.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            pred = model(img)

            pred = torch.nn.functional.interpolate(
                pred,
                size=depth.shape[2:],
                mode='bilinear',
                align_corners=False
            )

            loss = criterion(pred, depth)

        if torch.isnan(loss):
            print("Skipping NaN batch")
            continue

        scaler.scale(loss).backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch} Loss: {avg_loss}")

    torch.save(model.state_dict(),
               f"{save_path}/model_{epoch}.pth")

/tmp/ipykernel_310/308618347.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(
  0%|          | 0/12672 [00:00<?, ?it/s]/tmp/ipykernel_310/308618347.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(
  0%|          | 0/12672 [00:10<?, ?it/s]


RuntimeError: Sizes of tensors must match except in dimension 1. Expected size 16 but got size 15 for tensor number 1 in the list.